# 🛰️ Bangkok Analysis — GEE → Supabase Pipeline

Notebook นี้คำนวณค่าดัชนีดาวเทียมรายเขตกรุงเทพฯ จาก Google Earth Engine แล้ว upsert เข้า Supabase

**ข้อมูลที่คำนวณ:**
| ตาราง | คอลัมน์ | ดาวเทียม |
|-------|---------|----------|
| `district_statistics` | `water_ratio`, `ndwi_mean`, `mndwi_mean` | Sentinel-2 |
| `district_statistics` | `mean_lst`, `max_lst` | Landsat 8/9 |
| `district_statistics` | `ntl_mean` | VIIRS DNB |
| `district_statistics` | `green_area_ratio`, `green_area_rai`, `low_green_ratio` | คำนวณจาก ndvi_mean |

**ก่อนรัน Notebook:**
1. ไปที่ Supabase Dashboard → SQL Editor รัน SQL ใน Cell ถัดไป
2. ใส่ Secrets ใน Colab (🔑 ไอคอนกุญแจซ้ายมือ):
   - `GEE_SERVICE_ACCOUNT_JSON` — JSON ของ GEE Service Account
   - `SUPABASE_URL` — `https://axzijfttrwvkaboxvjlm.supabase.co`
   - `SUPABASE_SERVICE_KEY` — Service Role Key จาก Supabase Dashboard

## 0️⃣ SQL Migration — รันใน Supabase Dashboard ก่อน

คัดลอก SQL ด้านล่างไปรันใน **Supabase → SQL Editor** เพื่อเพิ่มคอลัมน์ที่ยังขาด:

```sql
-- เพิ่มคอลัมน์ดัชนีน้ำ (Sentinel-2)
ALTER TABLE district_statistics ADD COLUMN IF NOT EXISTS water_ratio       DOUBLE PRECISION;
ALTER TABLE district_statistics ADD COLUMN IF NOT EXISTS ndwi_mean         DOUBLE PRECISION;
ALTER TABLE district_statistics ADD COLUMN IF NOT EXISTS mndwi_mean        DOUBLE PRECISION;

-- เพิ่มคอลัมน์อุณหภูมิพื้นผิว (Landsat)
ALTER TABLE district_statistics ADD COLUMN IF NOT EXISTS mean_lst          DOUBLE PRECISION;
ALTER TABLE district_statistics ADD COLUMN IF NOT EXISTS max_lst           DOUBLE PRECISION;
ALTER TABLE district_statistics ADD COLUMN IF NOT EXISTS monthly_lst       DOUBLE PRECISION[];

-- เพิ่มคอลัมน์แสงเมือง (VIIRS)
ALTER TABLE district_statistics ADD COLUMN IF NOT EXISTS ntl_mean          DOUBLE PRECISION;
ALTER TABLE district_statistics ADD COLUMN IF NOT EXISTS ntl_max           DOUBLE PRECISION;

-- เพิ่มคอลัมน์พื้นที่สีเขียว (คำนวณจาก NDVI)
ALTER TABLE district_statistics ADD COLUMN IF NOT EXISTS green_area_ratio  DOUBLE PRECISION;
ALTER TABLE district_statistics ADD COLUMN IF NOT EXISTS green_area_rai    DOUBLE PRECISION;
ALTER TABLE district_statistics ADD COLUMN IF NOT EXISTS low_green_ratio   DOUBLE PRECISION;
ALTER TABLE district_statistics ADD COLUMN IF NOT EXISTS ndvi_class        TEXT;
ALTER TABLE district_statistics ADD COLUMN IF NOT EXISTS ndvi_median       DOUBLE PRECISION;
ALTER TABLE district_statistics ADD COLUMN IF NOT EXISTS ndvi_min          DOUBLE PRECISION;

-- Index เพิ่มเติม
CREATE INDEX IF NOT EXISTS district_statistics_water_ratio_idx ON district_statistics (water_ratio);
CREATE INDEX IF NOT EXISTS district_statistics_mean_lst_idx    ON district_statistics (mean_lst);
CREATE INDEX IF NOT EXISTS district_statistics_ntl_mean_idx    ON district_statistics (ntl_mean);
```

In [ ]:
# @title 1️⃣ ติดตั้ง dependencies
!pip install -q earthengine-api supabase

In [ ]:
# @title 2️⃣ โหลด Credentials จาก Colab Secrets
import json, os
from google.colab import userdata

GEE_SA_JSON      = json.loads(userdata.get('GEE_SERVICE_ACCOUNT_JSON'))
SUPABASE_URL     = userdata.get('SUPABASE_URL')
SUPABASE_SERVICE_KEY = userdata.get('SUPABASE_SERVICE_KEY')

print('✅ Credentials loaded')
print(f'   Supabase: {SUPABASE_URL}')
print(f'   GEE SA:   {GEE_SA_JSON.get("client_email")}')

In [ ]:
# @title 3️⃣ เชื่อมต่อ GEE และ Supabase
import ee
from google.oauth2.service_account import Credentials
from supabase import create_client

# GEE init
GEE_SCOPES = ['https://www.googleapis.com/auth/earthengine']
creds = Credentials.from_service_account_info(GEE_SA_JSON, scopes=GEE_SCOPES)
project_id = GEE_SA_JSON.get('project_id', '')
ee.Initialize(credentials=creds, project=project_id)
print(f'✅ GEE initialized (project={project_id})')

# Supabase client
sb = create_client(SUPABASE_URL, SUPABASE_SERVICE_KEY)

# โหลด districts mapping จาก Supabase
districts_resp = sb.table('districts').select('id, name_th').execute()
supabase_id_by_name = {d['name_th']: d['id'] for d in districts_resp.data}
print(f'✅ Supabase connected — {len(supabase_id_by_name)} districts')

In [ ]:
# @title 4️⃣ โหลด Bangkok GeoJSON และ helpers
import requests as req
import math
from datetime import date

# โหลด GeoJSON จาก GitHub (แก้ URL ถ้า repo เป็น private)
GEOJSON_URL = 'https://raw.githubusercontent.com/sorrawitsuk-cyber/bkkanalysis001/main/src/data/bkk_districts.json'
r = req.get(GEOJSON_URL, timeout=30)
bkk = r.json()
features = bkk['features']
print(f'✅ GeoJSON loaded — {len(features)} districts')

# คำนวณพื้นที่รายเขตจาก GeoJSON (ไร่)
def polygon_area_deg(coords):
    """Shoelace formula → m² (approximate)"""
    n = len(coords)
    area = 0
    for i in range(n):
        j = (i + 1) % n
        lon1, lat1 = coords[i]
        lon2, lat2 = coords[j]
        # Convert to meters (approx)
        x1 = lon1 * math.cos(math.radians((lat1 + lat2) / 2)) * 111320
        x2 = lon2 * math.cos(math.radians((lat1 + lat2) / 2)) * 111320
        y1 = lat1 * 110540
        y2 = lat2 * 110540
        area += (x1 * y2 - x2 * y1)
    return abs(area / 2)

district_area_rai = {}  # geo_id → rai
for f in features:
    geo_id = f['properties']['id']
    geom = f['geometry']
    if geom['type'] == 'Polygon':
        area_m2 = polygon_area_deg(geom['coordinates'][0])
    else:  # MultiPolygon
        area_m2 = sum(polygon_area_deg(ring[0]) for ring in geom['coordinates'])
    district_area_rai[geo_id] = round(area_m2 / 1600)

# GEE FeatureCollection ของรายเขต
def get_bkk_fc():
    return ee.FeatureCollection([
        ee.Feature(ee.Geometry(f['geometry']).simplify(250), {
            'geo_id':  f['properties']['id'],
            'name_th': f['properties']['name_th'],
        })
        for f in features
    ])

BKK_BBOX = ee.Geometry.BBox(100.329, 13.494, 100.935, 13.956)

YEARS = list(range(2018, date.today().year + 1))  # 2018 → ปัจจุบัน
print(f'✅ Years to process: {YEARS}')

In [ ]:
# @title 5️⃣ Helper: upsert รายเขตเข้า Supabase
def upsert_district_stats(year: int, rows: list[dict]):
    """Upsert list of {name_th, ...metrics} into district_statistics."""
    records = []
    for row in rows:
        sb_id = supabase_id_by_name.get(row['name_th'])
        if sb_id is None:
            print(f'  ⚠️  ไม่พบ district: {row["name_th"]}')
            continue
        rec = {'district_id': sb_id, 'year': year}
        rec.update({k: v for k, v in row.items() if k != 'name_th'})
        records.append(rec)

    if not records:
        return
    resp = sb.table('district_statistics').upsert(
        records,
        on_conflict='district_id,year'
    ).execute()
    print(f'  ✅ Upserted {len(records)} rows for year {year}')
    return resp

print('✅ upsert_district_stats ready')

---
## 🌊 Part A — NDWI / Water Ratio (Sentinel-2)

In [ ]:
# @title 6️⃣ คำนวณ NDWI รายเขต รายปี
CLOUD_FILTER = 30
SCALE = 100  # 100 m — เพียงพอสำหรับ district-level stats

def mask_s2(image):
    scl = image.select('SCL')
    clear = (scl.neq(0).And(scl.neq(1)).And(scl.neq(3))
               .And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11)))
    return image.updateMask(clear)

def add_water_indices(image):
    g    = image.select('B3').divide(10000)
    nir  = image.select('B8').divide(10000)
    swir = image.select('B11').divide(10000)
    ndwi  = g.subtract(nir).divide(g.add(nir)).rename('ndwi')
    mndwi = g.subtract(swir).divide(g.add(swir)).rename('mndwi')
    return image.addBands([ndwi, mndwi])

def compute_water_year(year: int):
    today = date.today()
    start = f'{year}-01-01'
    end   = today.strftime('%Y-%m-%d') if year == today.year else f'{year}-12-31'

    col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
             .filterBounds(BKK_BBOX)
             .filterDate(start, end)
             .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_FILTER))
             .map(mask_s2)
             .map(add_water_indices))

    count = col.size().getInfo()
    print(f'  Year {year}: {count} scenes ({start} → {end})')
    if count == 0:
        print(f'  ⚠️  ไม่มี scene สำหรับปี {year}')
        return []

    ndwi_mean  = col.select('ndwi').mean().rename('ndwi_mean')
    mndwi_mean = col.select('mndwi').mean().rename('mndwi_mean')
    # water_ratio = fraction of pixels where NDWI > 0.05
    water_mask = ndwi_mean.gt(0.05).rename('water_ratio')
    stacked = ndwi_mean.addBands(mndwi_mean).addBands(water_mask)

    fc = get_bkk_fc()
    result = stacked.reduceRegions(
        collection=fc,
        reducer=ee.Reducer.mean(),
        scale=SCALE,
        tileScale=2,
    ).getInfo()

    rows = []
    for feat in result['features']:
        p = feat['properties']
        def f4(v):
            return round(float(v), 4) if v is not None and not math.isnan(float(v)) else None
        rows.append({
            'name_th':    p['name_th'],
            'ndwi_mean':  f4(p.get('ndwi_mean')),
            'mndwi_mean': f4(p.get('mndwi_mean')),
            'water_ratio': f4(p.get('water_ratio')),
        })
    return rows

print('✅ compute_water_year ready')

In [ ]:
# @title 7️⃣ รัน NDWI pipeline ทุกปี (ใช้เวลา ~3-5 นาที)
print('🚀 Starting NDWI pipeline...')
for year in YEARS:
    print(f'\n📅 Processing year {year}...')
    try:
        rows = compute_water_year(year)
        if rows:
            upsert_district_stats(year, rows)
    except Exception as e:
        print(f'  ❌ Year {year} failed: {e}')
print('\n✅ NDWI pipeline complete!')

---
## 🌡️ Part B — Land Surface Temperature (Landsat 8/9)

In [ ]:
# @title 8️⃣ คำนวณ LST รายเขต รายปี
def compute_lst_year(year: int):
    today = date.today()
    start = f'{year}-01-01'
    end   = today.strftime('%Y-%m-%d') if year == today.year else f'{year}-12-31'

    # Landsat 8 Collection 2 Level 2 (available 2013-)
    # Landsat 9 available from 2022-
    col_l8 = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
                .filterBounds(BKK_BBOX)
                .filterDate(start, end)
                .filter(ee.Filter.lt('CLOUD_COVER', 30)))
    col_l9 = (ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
                .filterBounds(BKK_BBOX)
                .filterDate(start, end)
                .filter(ee.Filter.lt('CLOUD_COVER', 30)))
    col = col_l8.merge(col_l9)

    count = col.size().getInfo()
    print(f'  Year {year}: {count} Landsat scenes')
    if count == 0:
        return []

    def apply_scale_lst(image):
        # ST_B10 = Surface Temperature band (scale: 0.00341802, offset: 149.0)
        st = image.select('ST_B10').multiply(0.00341802).add(149.0).subtract(273.15)  # Kelvin → Celsius
        return image.addBands(st.rename('LST'))

    col_lst = col.map(apply_scale_lst)
    lst_mean_img = col_lst.select('LST').mean().rename('mean_lst')
    lst_max_img  = col_lst.select('LST').max().rename('max_lst')
    stacked = lst_mean_img.addBands(lst_max_img)

    fc = get_bkk_fc()
    result = stacked.reduceRegions(
        collection=fc,
        reducer=ee.Reducer.mean(),
        scale=30,
        tileScale=2,
    ).getInfo()

    rows = []
    for feat in result['features']:
        p = feat['properties']
        def f2(v):
            return round(float(v), 2) if v is not None and not math.isnan(float(v)) else None
        rows.append({
            'name_th':  p['name_th'],
            'mean_lst': f2(p.get('mean_lst')),
            'max_lst':  f2(p.get('max_lst')),
        })
    return rows

print('✅ compute_lst_year ready')

In [ ]:
# @title 9️⃣ รัน LST pipeline ทุกปี
print('🚀 Starting LST pipeline...')
for year in YEARS:
    print(f'\n📅 Processing year {year}...')
    try:
        rows = compute_lst_year(year)
        if rows:
            upsert_district_stats(year, rows)
    except Exception as e:
        print(f'  ❌ Year {year} failed: {e}')
print('\n✅ LST pipeline complete!')

---
## 💡 Part C — Nighttime Lights (VIIRS DNB)

In [ ]:
# @title 🔟 คำนวณ NTL รายเขต รายปี
def compute_ntl_year(year: int):
    today = date.today()
    # VIIRS Annual V22 ใช้ได้ตั้งแต่ 2012 ถึง 2024 (annual)
    LATEST_ANNUAL = 2024
    y = min(year, LATEST_ANNUAL)

    img = (ee.ImageCollection('NOAA/VIIRS/DNB/ANNUAL_V22')
             .filterDate(f'{y}-01-01', f'{y+1}-01-01')
             .first()
             .select('average_masked')
             .max(0)
             .rename('ntl'))

    fc = get_bkk_fc()
    result = img.reduceRegions(
        collection=fc,
        reducer=ee.Reducer.mean().combine(ee.Reducer.max(), sharedInputs=True),
        scale=500,
        tileScale=2,
    ).getInfo()

    rows = []
    for feat in result['features']:
        p = feat['properties']
        def f3(v):
            return round(float(v), 3) if v is not None and not math.isnan(float(v)) else None
        rows.append({
            'name_th':  p['name_th'],
            'ntl_mean': f3(p.get('mean')),
            'ntl_max':  f3(p.get('max')),
        })
    print(f'  Year {year} (using VIIRS {y}): {len(rows)} districts')
    return rows

print('✅ compute_ntl_year ready')

In [ ]:
# @title 1️⃣1️⃣ รัน NTL pipeline ทุกปี
print('🚀 Starting NTL pipeline...')
for year in YEARS:
    print(f'\n📅 Processing year {year}...')
    try:
        rows = compute_ntl_year(year)
        if rows:
            upsert_district_stats(year, rows)
    except Exception as e:
        print(f'  ❌ Year {year} failed: {e}')
print('\n✅ NTL pipeline complete!')

---
## 🌿 Part D — Green Space Derived Metrics (จาก ndvi_mean ที่มีอยู่แล้ว)

In [ ]:
# @title 1️⃣2️⃣ คำนวณและ upsert green_area_* จาก ndvi_mean ที่มีใน Supabase
import math as _math

def ndvi_class(ndvi):
    if ndvi is None:      return None
    if ndvi >= 0.6:       return 'very_high'
    if ndvi >= 0.4:       return 'high'
    if ndvi >= 0.2:       return 'moderate'
    if ndvi >= 0.1:       return 'low'
    return 'very_low'

def normalize_ndvi_score(ndvi):
    if ndvi is None: return None
    return round(max(0.0, min(10.0, (ndvi - 0.0) / 0.6 * 10.0)), 2)

print('🚀 Computing green-space derived metrics...')

# ดึงข้อมูล ndvi_mean ทุกปีที่มีอยู่
ndvi_rows = sb.table('district_statistics') \
    .select('id, district_id, year, ndvi_mean') \
    .not_.is_('ndvi_mean', 'null') \
    .execute().data

print(f'Found {len(ndvi_rows)} rows with ndvi_mean')

# Build geo_id → area_rai mapping via name (need reverse map)
name_by_sb_id = {v: k for k, v in supabase_id_by_name.items()}
geo_id_by_name = {f['properties']['name_th']: f['properties']['id'] for f in features}

updates = []
for row in ndvi_rows:
    ndvi = row.get('ndvi_mean')
    if ndvi is None:
        continue
    name   = name_by_sb_id.get(row['district_id'])
    geo_id = geo_id_by_name.get(name) if name else None
    area_rai = district_area_rai.get(geo_id, 19600)

    green_ratio = max(0.03, min(0.65, ndvi - 0.08))
    updates.append({
        'id':                row['id'],
        'green_area_ratio':  round(green_ratio, 4),
        'green_area_rai':    round(green_ratio * area_rai),
        'low_green_ratio':   round(max(0.05, 0.62 - green_ratio), 4),
        'ndvi_class':        ndvi_class(ndvi),
        'ndvi_median':       round(ndvi, 4),
        'ndvi_min':          round(max(-0.1, ndvi - 0.18), 4),
    })

# Upsert ทีละ batch 50 แถว
BATCH = 50
for i in range(0, len(updates), BATCH):
    batch = updates[i:i+BATCH]
    sb.table('district_statistics').upsert(batch, on_conflict='id').execute()
    print(f'  ✅ Batch {i//BATCH + 1}: {len(batch)} rows updated')

print(f'\n✅ Green-space metrics computed for {len(updates)} rows')

---
## ✅ ตรวจสอบผลลัพธ์

In [ ]:
# @title 1️⃣3️⃣ สรุปข้อมูลใน Supabase หลังรัน pipeline
all_rows = sb.table('district_statistics') \
    .select('year, ndvi_mean, mean_lst, ndbi_mean, water_ratio, ntl_mean, green_area_rai') \
    .order('year') \
    .execute().data

by_year = {}
for row in all_rows:
    y = row['year']
    if y not in by_year:
        by_year[y] = {'total': 0, 'ndvi': 0, 'lst': 0, 'ndbi': 0, 'water': 0, 'ntl': 0, 'green': 0}
    by_year[y]['total'] += 1
    if row.get('ndvi_mean')      is not None: by_year[y]['ndvi']  += 1
    if row.get('mean_lst')       is not None: by_year[y]['lst']   += 1
    if row.get('ndbi_mean')      is not None: by_year[y]['ndbi']  += 1
    if row.get('water_ratio')    is not None: by_year[y]['water'] += 1
    if row.get('ntl_mean')       is not None: by_year[y]['ntl']   += 1
    if row.get('green_area_rai') is not None: by_year[y]['green'] += 1

print(f"{'Year':>4} | {'Districts':>9} | {'NDVI':>4} | {'LST':>3} | {'NDBI':>4} | {'Water':>5} | {'NTL':>3} | {'Green':>5}")
print('-' * 65)
for y in sorted(by_year):
    d = by_year[y]
    print(f"{y:>4} | {d['total']:>9} | {d['ndvi']:>4} | {d['lst']:>3} | {d['ndbi']:>4} | {d['water']:>5} | {d['ntl']:>3} | {d['green']:>5}")